# Stage 1 (~100k trainable parameters): PANNs CNN14 + residual embedding adapter

This notebook builds on the successful classifier-head-only baseline and adds a small trainable residual adapter after the pretrained CNN14 embedding.

- Task: binary ICBHI classification, **Normal (0)** vs **Abnormal (1)**.
- The original AudioSet-pretrained CNN14 backbone stays frozen and in evaluation mode.
- A bottleneck adapter and binary classifier are trainable: **100,377 parameters** with the default dimensions.
- Train/validation splitting is patient-level and uses only official training patients.
- The official test split is not used for checkpoint selection; best checkpoint is selected by validation HS-score, and the last epoch is saved separately.

**Why an adapter?** Unfreezing a native CNN14 convolution layer can jump the trainable parameter count by millions. A bottleneck adapter allows a controlled first step near 100k trainable parameters.

**Dataset assumptions:** `split_metadata.csv` has `source_file`, `start`, `end`, `crackle`, `wheeze`, and `split` columns. `source_file` follows the ICBHI convention where patient ID is the part before the first underscore. `Cnn14_mAP=0.431.pth` must be compatible with the local `Cnn14` implementation.

Run the notebook from a fresh kernel. It loads the original pretrained checkpoint and does not resume from checkpoints created by the earlier unstable all-backbone fine-tuning run.


In [1]:
# 1. Imports and reproducibility
import os
import random
import json
import importlib.util
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score


D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#from audioset_tagging_cnn_master.pytorch.models import Cnn14
SRC_DIR = Path.cwd()
PYTORCH_DIR = SRC_DIR / "audioset_tagging_cnn_master" / "pytorch"
MODELS_PATH = PYTORCH_DIR / "models.py"
LOCAL_UTILS_PATH = PYTORCH_DIR / "pytorch_utils.py"

assert MODELS_PATH.is_file(), f"Missing file: {MODELS_PATH}"
assert LOCAL_UTILS_PATH.is_file(), f"Missing file: {LOCAL_UTILS_PATH}"

# Make the local helper available under the name expected by AudioSet.
utils_spec = importlib.util.spec_from_file_location(
    "pytorch_utils", LOCAL_UTILS_PATH
)
utils_module = importlib.util.module_from_spec(utils_spec)
sys.modules["pytorch_utils"] = utils_module


# Execute the local helper module
utils_spec.loader.exec_module(utils_module)

print("Local helper loaded:", utils_module.__file__)
print("Has forward:", hasattr(utils_module, "forward"))

# Load models.py directly, without executing the package __init__.py
models_spec = importlib.util.spec_from_file_location(
    "audioset_cnn14_models", MODELS_PATH
)
models_module = importlib.util.module_from_spec(models_spec)
sys.modules[models_spec.name] = models_module
models_spec.loader.exec_module(models_module)

Cnn14 = models_module.Cnn14
print("CNN14 imported successfully:", Cnn14)



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Local helper loaded: D:\.JKU CS courses\Master Thesis\src\audioset_tagging_cnn_master\pytorch\pytorch_utils.py
Has forward: True
CNN14 imported successfully: <class 'audioset_cnn14_models.Cnn14'>
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [3]:
# 2. Paths and hyperparameters — update paths for your local project
AUDIO_DIR = Path("../datasets/raw/ICBHI_final_database/audios")
METADATA_PATH = Path("../datasets/split_metadata.csv")
PRETRAINED_CHECKPOINT = Path("Cnn14_mAP=0.431.pth")
OUTPUT_DIR = Path("../models/cnn14_stage1_100k_adapter")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 10.0  # fixed input length for batching; segments are cropped using metadata start/end
BATCH_SIZE = 8
NUM_WORKERS = 0  # safe default on Windows/Jupyter
EPOCHS = 20
VAL_SIZE = 0.20
CLASSIFIER_LR = 1e-3
ADAPTER_LR = 3e-4
WEIGHT_DECAY = 1e-4
ADAPTER_DIM = 23  # yields 100,377 trainable parameters including the classifier
USE_AMP = False
RUN_FINAL_TEST_EVALUATION = False  # keep False while comparing training stages

BEST_CHECKPOINT = OUTPUT_DIR / "cnn14_best_val_hs.pt"
LAST_CHECKPOINT = OUTPUT_DIR / "cnn14_last_epoch.pt"
RESULTS_PATH = OUTPUT_DIR / "test_results.json"

print("Audio directory:", AUDIO_DIR.resolve())
print("Metadata:", METADATA_PATH.resolve())
print("Pretrained checkpoint:", PRETRAINED_CHECKPOINT.resolve())

Audio directory: D:\.JKU CS courses\Master Thesis\datasets\raw\ICBHI_final_database\audios
Metadata: D:\.JKU CS courses\Master Thesis\datasets\split_metadata.csv
Pretrained checkpoint: D:\.JKU CS courses\Master Thesis\src\Cnn14_mAP=0.431.pth


In [4]:
# 3. Load and validate metadata
for required_path in (AUDIO_DIR, METADATA_PATH, PRETRAINED_CHECKPOINT):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {required_path.resolve()}\n"
            "Update AUDIO_DIR, METADATA_PATH, and PRETRAINED_CHECKPOINT to match your local files."
        )

df = pd.read_csv(METADATA_PATH)
required_columns = {"source_file", "start", "end", "crackle", "wheeze", "split"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing)}")

df = df.copy()
df["source_file"] = df["source_file"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip().str.lower()
split_map = {"train": "train", "training": "train", "test": "test", "testing": "test"}
unknown = sorted(set(df["split"]) - set(split_map))
if unknown:
    raise ValueError(f"Unexpected split values {unknown}; expected official train/test labels.")
df["split"] = df["split"].map(split_map)

def patient_id_from_source(source_file):
    stem = Path(source_file).stem
    if "_" not in stem:
        raise ValueError(
            f"Cannot infer patient ID from '{source_file}'. Expected ICBHI recording filename "
            "with patient ID before the first underscore."
        )
    return stem.split("_", 1)[0]

df["patient_id"] = df["source_file"].map(patient_id_from_source)
if df[list(required_columns)].isna().any().any():
    raise ValueError("Metadata contains missing values in required columns.")

official_train = df[df["split"] == "train"].copy()
official_test = df[df["split"] == "test"].copy()
if official_train.empty or official_test.empty:
    raise ValueError("Both official train and official test rows must be present.")

""" 
since there is data overlap in the official data split, the error << raise ValueError() >>
will be ignored and replaced with a warning message << warnings.warn() >>
"""
overlap = set(official_train.patient_id) & set(official_test.patient_id)
if overlap:
    warnings.warn(
        f"\n!!! WARNING DATA LEAKAGE !!!\nPatient leakage in metadata: {len(overlap)} patient(s) appear in "
        f"both official train and test, e.g. {sorted(overlap)[:10]}. Correct split_metadata.csv before training!"
    )

print(f"Official train: {len(official_train):,} segments / {official_train.patient_id.nunique()} patients")
print(f"Official test:  {len(official_test):,} segments / {official_test.patient_id.nunique()} patients")

Official train: 4,142 segments / 79 patients
Official test:  2,756 segments / 49 patients


C:\Users\arebr\AppData\Local\Temp\ipykernel_25300\2842253417.py:48: UserWarning: 
!!! WARNING DATA LEAKAGE !!!
Patient leakage in metadata: 2 patient(s) appear in both official train and test, e.g. ['156', '218']. Correct split_metadata.csv before training!
  warnings.warn(


In [5]:
# 4. Patient-level validation split from official training data only

splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)

tr_idx, va_idx = next(splitter.split(
                                    X=np.zeros(len(official_train)),
                                    y=official_train["label"],
                                    groups=official_train["patient_id"]
                                    )
)

train_df = official_train.iloc[tr_idx].reset_index(drop=True)
val_df = official_train.iloc[va_idx].reset_index(drop=True)
test_df = official_test.reset_index(drop=True)  # Keep official test untouched

# Verify patient separation
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])

assert train_patients.isdisjoint(val_patients), "Train/validation leakage!"

# Binary labels: 0 = Normal, 1 = Abnormal
def binary_label(crackle, wheeze):
    return 0 if int(crackle) == 0 and int(wheeze) == 0 else 1

for frame in (train_df, val_df, test_df):
    frame["label"] = [
        binary_label(c, w)
        for c, w in zip(frame["crackle"], frame["wheeze"])
    ]

# Split sizes and patient counts
display(pd.DataFrame([
    {
        "split": "train",
        "segments": len(train_df),
        "patients": train_df["patient_id"].nunique()
    },
    {
        "split": "validation",
        "segments": len(val_df),
        "patients": val_df["patient_id"].nunique()
    },
    {
        "split": "official_test",
        "segments": len(test_df),
        "patients": test_df["patient_id"].nunique()
    }
]))

# Compare class distributions
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Official test", test_df)
]:
    print(f"\n{name} class counts:")
    print(frame["label"].value_counts().reindex([0, 1], fill_value=0))
    print(f"{name} class proportions:")
    print(frame["label"].value_counts(normalize=True).reindex(
        [0, 1], fill_value=0
    ).round(4))

print("\nNo patient overlap between train and validation.")


,split,segments,patients
0,train,3485,61
1,validation,657,18
2,official_test,2756,49



Train class counts:
label
0    1788
1    1697
Name: count, dtype: int64
Train class proportions:
label
0    0.5131
1    0.4869
Name: proportion, dtype: float64

Validation class counts:
label
0    275
1    382
Name: count, dtype: int64
Validation class proportions:
label
0    0.4186
1    0.5814
Name: proportion, dtype: float64

Official test class counts:
label
0    1579
1    1177
Name: count, dtype: int64
Official test class proportions:
label
0    0.5729
1    0.4271
Name: proportion, dtype: float64

No patient overlap between train and validation.


In [6]:
# 5. Audio dataset: crop each annotated segment, resample to 16 kHz, pad/trim to fixed duration
class ICBHIAudioDataset(Dataset):
    def __init__(self, frame, audio_dir):
        self.frame = frame.reset_index(drop=True).copy()
        self.audio_dir = Path(audio_dir)
        self.cache = {}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        filename = row["source_file"]
        if not filename.lower().endswith(".wav"):
            filename += ".wav"
        wav_path = self.audio_dir / filename
        if not wav_path.exists():
            raise FileNotFoundError(f"Audio file not found: {wav_path}")

        if filename not in self.cache:
            audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            self.cache[filename] = audio
        audio = self.cache[filename]

        start = max(0, int(float(row["start"]) * SAMPLE_RATE))
        end = min(len(audio), int(float(row["end"]) * SAMPLE_RATE))
        segment = audio[start:end]
        if len(segment) == 0:
            raise ValueError(f"Empty segment for {filename}: start={row['start']}, end={row['end']}")

        target_len = int(SAMPLE_RATE * SEGMENT_SECONDS)
        if len(segment) < target_len:
            segment = np.pad(segment, (0, target_len - len(segment)))
        else:
            segment = segment[:target_len]

        # Return waveform [time] and integer class label.
        return torch.tensor(segment, dtype=torch.float32), torch.tensor(int(row["label"]), dtype=torch.long)

train_ds = ICBHIAudioDataset(train_df, AUDIO_DIR)
val_ds = ICBHIAudioDataset(val_df, AUDIO_DIR)
test_ds = ICBHIAudioDataset(test_df, AUDIO_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=(device.type == "cuda"))

wave_batch, label_batch = next(iter(train_loader))
print("Waveform batch:", tuple(wave_batch.shape), "label batch:", tuple(label_batch.shape))

Waveform batch: (8, 160000) label batch: (8,)


In [7]:
# 6. Initialize pretrained CNN14 and add a small trainable residual adapter
# Load the original 527-class AudioSet model so checkpoint dimensions match.
backbone = Cnn14(
    sample_rate=16000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=8000,
    classes_num=527,
)

checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
print("Pretrained checkpoint loaded.")
print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")

class CNN14BinaryClassifier(nn.Module):
    def __init__(self, cnn14_backbone, embedding_dim=2048, num_classes=2, adapter_dim=23):
        super().__init__()
        self.backbone = cnn14_backbone
        # Bottleneck MLP adapter: 2048 -> 23 -> 2048.
        # Its final layer is zero-initialized, so the adapter initially acts as an identity residual.
        self.embedding_adapter = nn.Sequential(
            nn.Linear(embedding_dim, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, embedding_dim),
        )
        nn.init.zeros_(self.embedding_adapter[-1].weight)
        nn.init.zeros_(self.embedding_adapter[-1].bias)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, waveform):
        outputs = self.backbone(waveform, None)
        embedding = outputs["embedding"]
        adapted_embedding = embedding + self.embedding_adapter(embedding)
        logits = self.classifier(adapted_embedding)
        return logits

model = CNN14BinaryClassifier(backbone, adapter_dim=ADAPTER_DIM).to(device)

# Freeze the pretrained backbone. Only the adapter and binary head are trainable.
for parameter in model.backbone.parameters():
    parameter.requires_grad = False
for parameter in model.embedding_adapter.parameters():
    parameter.requires_grad = True
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
expected_parameters = (2048 * ADAPTER_DIM + ADAPTER_DIM) + (ADAPTER_DIM * 2048 + 2048) + (2048 * 2 + 2)
print(f"Adapter bottleneck dimension: {ADAPTER_DIM}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Expected trainable parameters: {expected_parameters:,}")
assert trainable_parameters == expected_parameters, "Unexpected adapter/head parameter count."
assert 90_000 <= trainable_parameters <= 110_000, "Trainable parameter count is not near the 100k target."


Pretrained checkpoint loaded.
Missing keys: [] 
Unexpected keys: [] 
Adapter bottleneck dimension: 23
Trainable parameters: 100,377
Expected trainable parameters: 100,377


In [8]:
# 7. Metrics and epoch runner
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    hs = (2 * sensitivity * specificity / (sensitivity + specificity)
          if (sensitivity + specificity) else 0.0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "HS_score": float(hs),
        "confusion_matrix": cm.tolist(),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
        {"params": model.embedding_adapter.parameters(), "lr": ADAPTER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(model, loader, training):
    model.train(training)
    # The whole pretrained backbone is frozen: keep it in eval mode so its
    # BatchNorm running statistics and any backbone dropout behavior stay fixed.
    model.backbone.eval()
                
    total_loss = 0.0
    y_true, y_pred = [], []

    for waveforms, labels in tqdm(loader, leave=False):
        waveforms = waveforms.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")
                               ):
                logits = model(waveforms)
            
                # Check model outputs before calculating the loss
                if not torch.isfinite(logits).all():
                    raise RuntimeError(f"Non-finite logits detected. Training={training}")
            
                loss = criterion(logits, labels)
            
            # Check loss BEFORE backward() or optimizer.step()
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss detected. "
                    f"Training={training}, loss={loss.item()}"
                )


            if training:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()


        total_loss += loss.item() * waveforms.size(0)
        predictions = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(predictions.detach().cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(loader.dataset))
    return metrics

In [9]:
model.eval()

waveforms, labels = next(iter(train_loader))
waveforms = waveforms.to(device)

with torch.no_grad():
    logits = model(waveforms)

print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())

bad_buffers = [
    name for name, buf in model.named_buffers()
    if torch.is_floating_point(buf)
    and not torch.isfinite(buf).all()
]
print("Invalid model buffers:", bad_buffers)

if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Fresh model still produces NaN/Inf logits. Do not start training."
    )

print("Full-precision forward pass passed.")


Waveforms finite: True
Logits shape: (8, 2)
Logits finite: True
Invalid model buffers: []
Full-precision forward pass passed.


In [10]:
batch = next(iter(train_loader))
waveforms, labels = batch[:2]

print("Waveform shape:", waveforms.shape)
print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Waveform min/max:", waveforms.min().item(), waveforms.max().item())
print("Labels:", torch.unique(labels))
print("Labels finite:", torch.isfinite(labels).all().item())

Waveform shape: torch.Size([8, 160000])
Waveforms finite: True
Waveform min/max: -1.0058673620224 1.00046706199646
Labels: tensor([0, 1])
Labels finite: True


In [11]:
# 8. Train and save best/last checkpoints
history = []
best_val_hs = -1.0
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)
    scheduler.step(val_metrics["HS_score"])
    classifier_lr_now = optimizer.param_groups[0]["lr"]
    adapter_lr_now = optimizer.param_groups[1]["lr"]

    history.append({
        "epoch": epoch,
        "classifier_lr": classifier_lr_now,
        "adapter_lr": adapter_lr_now,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_sensitivity": train_metrics["sensitivity"],
        "train_specificity": train_metrics["specificity"],
        "train_HS_score": train_metrics["HS_score"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_sensitivity": val_metrics["sensitivity"],
        "val_specificity": val_metrics["specificity"],
        "val_HS_score": val_metrics["HS_score"],
    })

    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_metrics": val_metrics,
        "seed": SEED,
        "class_mapping": {"Normal": 0, "Abnormal": 1},
        "config": {
            "sample_rate": SAMPLE_RATE,
            "segment_seconds": SEGMENT_SECONDS,
            "stage": "stage1_100k_adapter",
            "adapter_dim": ADAPTER_DIM,
            "trainable_parameters": trainable_parameters,
            "classifier_lr": classifier_lr_now,
            "adapter_lr": adapter_lr_now,
            "batch_size": BATCH_SIZE,
        },
    }
    torch.save(payload, LAST_CHECKPOINT)

    if val_metrics["HS_score"] > best_val_hs:
        best_val_hs = val_metrics["HS_score"]
        best_epoch = epoch
        torch.save(payload, BEST_CHECKPOINT)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
        f"HS={train_metrics['HS_score']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
        f"SE={val_metrics['sensitivity']:.4f}, SP={val_metrics['specificity']:.4f}, "
        f"HS={val_metrics['HS_score']:.4f} | "
        f"classifier_lr={classifier_lr_now:.2e}, adapter_lr={adapter_lr_now:.2e}"
    )

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Best epoch:", best_epoch, "| best validation HS:", round(best_val_hs, 4))
print("Best checkpoint:", BEST_CHECKPOINT.resolve())
print("Last checkpoint:", LAST_CHECKPOINT.resolve())

Epoch 01/20 | train loss=0.6600, acc=0.6023, HS=0.6023 | val loss=0.6635, acc=0.5677, SE=0.2880, SP=0.9564, HS=0.4426 | classifier_lr=1.00e-03, adapter_lr=3.00e-04


Epoch 02/20 | train loss=0.6321, acc=0.6373, HS=0.6370 | val loss=0.6107, acc=0.6606, SE=0.5209, SP=0.8545, HS=0.6473 | classifier_lr=1.00e-03, adapter_lr=3.00e-04


Epoch 03/20 | train loss=0.6222, acc=0.6416, HS=0.6417 | val loss=0.6881, acc=0.5616, SE=0.2513, SP=0.9927, HS=0.4011 | classifier_lr=1.00e-03, adapter_lr=3.00e-04


Epoch 04/20 | train loss=0.6046, acc=0.6657, HS=0.6654 | val loss=0.5664, acc=0.7032, SE=0.7435, SP=0.6473, HS=0.6920 | classifier_lr=1.00e-03, adapter_lr=3.00e-04


Epoch 05/20 | train loss=0.5951, acc=0.6801, HS=0.6798 | val loss=0.5477, acc=0.6925, SE=0.8665, SP=0.4509, HS=0.5932 | classifier_lr=1.00e-03, adapter_lr=3.00e-04


...

Epoch 18/20 | train loss=0.5429, acc=0.7222, HS=0.7216 | val loss=0.5445, acc=0.7275, SE=0.7173, SP=0.7418, HS=0.7293 | classifier_lr=5.00e-04, adapter_lr=1.50e-04


Epoch 19/20 | train loss=0.5447, acc=0.7277, HS=0.7276 | val loss=0.5803, acc=0.7291, SE=0.6152, SP=0.8873, HS=0.7266 | classifier_lr=2.50e-04, adapter_lr=7.50e-05


Epoch 20/20 | train loss=0.5383, acc=0.7300, HS=0.7298 | val loss=0.5424, acc=0.7184, SE=0.7173, SP=0.7200, HS=0.7186 | classifier_lr=2.50e-04, adapter_lr=7.50e-05


,epoch,classifier_lr,adapter_lr,train_loss,train_accuracy,train_sensitivity,train_specificity,train_HS_score,val_loss,val_accuracy,val_sensitivity,val_specificity,val_HS_score
0,1,0.00100,0.000300,0.660033,0.602296,0.601650,0.602908,0.602278,0.663527,0.567732,0.287958,0.956364,0.442639
1,2,0.00100,0.000300,0.632095,0.637303,0.629935,0.644295,0.637034,0.610698,0.660578,0.520942,0.854545,0.647289
2,3,0.00100,0.000300,0.622248,0.641607,0.648203,0.635347,0.641710,0.688073,0.561644,0.251309,0.992727,0.401084
3,4,0.00100,0.000300,0.604568,0.665710,0.657042,0.673937,0.665382,0.566418,0.703196,0.743455,0.647273,0.692038
4,5,0.00100,0.000300,0.595072,0.680057,0.672952,0.686801,0.679806,0.547719,0.692542,0.866492,0.450909,0.593151
5,6,0.00100,0.000300,0.586784,0.685222,0.676488,0.693512,0.684894,0.551773,0.716895,0.722513,0.709091,0.715739
6,7,0.00100,0.000300,0.580447,0.687805,0.684738,0.690716,0.687714,0.546350,0.716895,0.751309,0.669091,0.707820
7,8,0.00100,0.000300,0.585542,0.688953,0.681202,0.696309,0.688673,0.537994,0.706240,0.780105,0.603636,0.680618
8,9,0.00100,0.000300,0.573906,0.695839,0.688273,0.703020,0.695569,0.540813,0.695586,0.848168,0.483636,0.616014
9,10,0.00100,0.000300,0.566025,0.702439,0.690631,0.713647,0.701950,0.554801,0.727549,0.688482,0.781818,0.732187


Best epoch: 11 | best validation HS: 0.7446
Best checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_stage1_100k_adapter\cnn14_best_val_hs.pt
Last checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_stage1_100k_adapter\cnn14_last_epoch.pt




### Stage 1 is now complete. 
We trained the CNN14-based binary classification model using an adapter with approximately **100,000 trainable parameters** and selected the best checkpoint based on the validation HS-score.
<br><br>
### Next Step: Stage 2 — Training with 1 Million Trainable Parameters

#### What happens next?

1. Open the next notebook: **`"05.6. cnn14_end_to_end__1m_parameters.ipynb"`**.
3. Follow the notebook cells to train **Stage 2**, using an adapter with approximately **1 million trainable parameters**.
4. After Stage 2 training is complete, evaluate the best checkpoints from **both Stage 1 and Stage 2** on the official test set.
5. Compare their final performance using loss, accuracy, sensitivity, specificity, HS-score, and confusion matrices.

The final comparison will help us assess whether increasing the adapter's capacity from approximately 100K to 1M trainable parameters improves binary respiratory sound classification.



### Optional: Evaluate Stage 1 only

If evaluation of Stage 1 on the official test set is only wanted without waiting for Stage 2, uncomment the following line in the next code cell:

```python
RUN_FINAL_TEST_EVALUATION = True
```

Then run the following cells to perform the Stage 1 test evaluation.

**Important:** The official test set is reserved for final evaluation. Do not use its results to tune model parameters or select checkpoints.


In [1]:
# RUN_FINAL_TEST_EVALUATION = True

In [12]:
# 9. Final official test evaluation
def evaluate_checkpoint(checkpoint_path, loader, label):
    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
    eval_backbone = Cnn14(
        sample_rate=16000, window_size=1024, hop_size=320,
        mel_bins=64, fmin=50, fmax=8000, classes_num=527
    )
    eval_model = CNN14BinaryClassifier(eval_backbone).to(device)
    eval_model.load_state_dict(payload["model_state_dict"])

    # Evaluate without optimizer updates.
    eval_model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for waveforms, labels in tqdm(loader, leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = eval_model(waveforms)
            loss = criterion(logits, labels)
            total_loss += loss.item() * waveforms.size(0)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(loader.dataset))
    print(f"\n{label} checkpoint — epoch {payload['epoch']}")
    print(f"Test loss:   {metrics['loss']:.4f}")
    print(f"Accuracy:    {metrics['accuracy']:.4f}")
    print(f"Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"Specificity: {metrics['specificity']:.4f}")
    print(f"HS-score:    {metrics['HS_score']:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(metrics["confusion_matrix"]))

    return {
        "checkpoint": str(checkpoint_path),
        "epoch": int(payload["epoch"]),
        "test_loss": metrics["loss"],
        "test_accuracy": metrics["accuracy"],
        "test_sensitivity": metrics["sensitivity"],
        "test_specificity": metrics["specificity"],
        "test_HS_score": metrics["HS_score"],
        "confusion_matrix": metrics["confusion_matrix"],
    }

if RUN_FINAL_TEST_EVALUATION:
    best_test = evaluate_checkpoint(BEST_CHECKPOINT, test_loader, "BEST validation-HS")
    last_test = evaluate_checkpoint(LAST_CHECKPOINT, test_loader, "LAST epoch")
else:
    best_test = None
    last_test = None
    print(
        "Official test evaluation skipped. Keep it untouched while comparing stages; "
        "set RUN_FINAL_TEST_EVALUATION=True only for the final selected model."
    )

Official test evaluation skipped. Keep it untouched while comparing stages; set RUN_FINAL_TEST_EVALUATION=True only for the final selected model.


In [ ]:
# 10. Save test results and a compact comparison table
results = {
    "experiment_stage": "stage1_100k_adapter",
    "trainable_parameters": int(trainable_parameters),
    "adapter_dim": int(ADAPTER_DIM),
    "checkpoint_selection": "highest validation HS-score",
    "best_epoch": int(best_epoch),
    "best_validation_HS_score": float(best_val_hs),
    "official_test_evaluated": bool(RUN_FINAL_TEST_EVALUATION),
    "best_checkpoint_test": best_test,
    "last_checkpoint_test": last_test,
    "split_counts": {
        "train_segments": int(len(train_df)),
        "validation_segments": int(len(val_df)),
        "official_test_segments": int(len(test_df)),
        "train_patients": int(train_df.patient_id.nunique()),
        "validation_patients": int(val_df.patient_id.nunique()),
        "official_test_patients": int(test_df.patient_id.nunique()),
    },
}
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

if RUN_FINAL_TEST_EVALUATION:
    display(pd.DataFrame([
        {"checkpoint": "best_validation_HS", **{k: best_test[k] for k in [
            "epoch", "test_loss", "test_accuracy", "test_sensitivity",
            "test_specificity", "test_HS_score"
        ]}},
        {"checkpoint": "last_epoch", **{k: last_test[k] for k in [
            "epoch", "test_loss", "test_accuracy", "test_sensitivity",
            "test_specificity", "test_HS_score"
        ]}},
    ]))
else:
    print("Test metrics were not calculated; validation metrics remain the basis for model selection.")
print("Saved results:", RESULTS_PATH.resolve())